In [1]:
from io import BytesIO
import numpy as np
import requests
import matplotlib.pyplot as plt
from scipy.ndimage import rotate
from torch.utils.data import TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
import pickle
import torchvision.transforms as T
from torchvision.transforms.functional import to_pil_image
import copy

In [ ]:
#Icon loading and initial verification
if 'icons' not in locals():
  !wget https://www.dropbox.com/s/jgjuf2t0enioz1n/iconos_train.pkl
  with open('/content/iconos_train.pkl', 'rb') as handle:
      icons = pickle.load(handle)
print(icons.keys())
icons['images'].shape

In [ ]:
#First 100 images visualization
number_of_icons_to_show = 100
fig = plt.figure(figsize=(20, 6))
for idx in np.arange(number_of_icons_to_show):
  ax = fig.add_subplot(5, 20, idx+1, xticks=[], yticks=[])
  ax.imshow(icons['images'][idx].transpose(2,1,0))
  ax.set_title(str(icons['labels'][idx]))

In [4]:
#After viewing the initial images, we can identify five categories
classes = ['Planes', 'Birds', 'Ships', 'Rabbits', 'Felines']

In [13]:
#Creation of training, validation and testing sets
collectionIndexes = np.arange(len(icons['images']))
np.random.shuffle(collectionIndexes) #Random indexes
n1 = int(len(collectionIndexes) * 0.6) #Last index of training set
n2 = int(len(collectionIndexes) * 0.8) #Last index of validation set
nf = int(len(collectionIndexes) * 0.75) #Division in case there is no testing set
trainIndexes = collectionIndexes[:nf]
validationIndexes = collectionIndexes[nf:]
testIndexes = collectionIndexes[n2:]
trainIcons = {
    'images': icons['images'][trainIndexes],
    'labels': icons['labels'][trainIndexes]
}
validationIcons = {
    'images': icons['images'][validationIndexes],
    'labels': icons['labels'][validationIndexes]
}
testIcons = {
    'images': icons['images'][testIndexes],
    'labels': icons['labels'][testIndexes]
}
#We need tensors to train the model
batch_size = 60
train_images = torch.tensor(trainIcons['images'], dtype = torch.float32) / 255
train_labels = torch.tensor(trainIcons['labels'], dtype = torch.uint8)

val_images = torch.tensor(validationIcons['images'], dtype = torch.float32) /255
val_labels = torch.tensor(validationIcons['labels'], dtype = torch.uint8)

test_images = torch.tensor(testIcons['images'], dtype = torch.float32) /255
test_labels = torch.tensor(testIcons['labels'], dtype = torch.uint8)
#DataLoaders are created from the tensors
train_loader = torch.utils.data.DataLoader(TensorDataset(train_images, train_labels), batch_size, True)
val_loader = torch.utils.data.DataLoader(TensorDataset(val_images, val_labels), batch_size, True)
test_loader = torch.utils.data.DataLoader(TensorDataset(test_images, test_labels), batch_size, True)

In [14]:
#Data augmentation for training set:
class AugmentedDataset(torch.utils.data.Dataset):
  def __init__(self, images, labels, transform=None):
    self.images = images
    self.labels = labels
    self.transform = transform

  def __len__(self):
    return len(self.images)

  def __getitem__(self, idx):
    img = to_pil_image(self.images[idx])
    label = self.labels[idx]
    if self.transform:
      img = self.transform(img)
    return img, label

#Image transformations:
transform_train = T.Compose([
    T.RandomHorizontalFlip(p = 0.5),
    T.RandomVerticalFlip(p = 0.2),
    T.RandomRotation(15),
    T.RandomAffine(degrees = 0, translate = (0.1, 0.1)),
    T.ColorJitter(0.2, 0.2),
    T.ToTensor()
])
#train_loader is overwritten to include data augmentation
train_loader = torch.utils.data.DataLoader(AugmentedDataset(train_images, train_labels, transform_train), batch_size, True)

In [ ]:
#Visualization of the first 20 images with data augmentation
dataiter = iter(train_loader)
images, labels = next(dataiter)
images = images.numpy()
fig = plt.figure(figsize = (20, 6))
for idx in np.arange(20):
    ax = fig.add_subplot(2, 10, idx+1, xticks=[], yticks=[])
    ax.imshow(np.squeeze(images[idx].transpose(2, 1, 0)))
    ax.set_title(str(labels[idx].item()))
plt.show()

In [ ]:
#Architecture design
class Net(nn.Module):
  def __init__(self, num_channels, num_classes):
    super(Net, self).__init__()
    self.conv1 = nn.Conv2d(num_channels, 32, 3, stride=1, padding=1)
    self.conv2 = nn.Conv2d(32, 64, 3, stride=1, padding=1)
    self.conv3 = nn.Conv2d(64, 64, 3, stride=1, padding=1)
    self.conv4 = nn.Conv2d(64, 128, 3, stride = 1, padding = 1)
    self.pool1 = nn.MaxPool2d(2)
    self.pool2 = nn.MaxPool2d(2)
    self.fc1 = nn.Linear(8 * 8 * 128, 128)
    self.fc2 = nn.Linear(128, num_classes)
    self.bn1 = nn.BatchNorm2d(32)
    self.bn2 = nn.BatchNorm2d(64)
    self.bn3 = nn.BatchNorm2d(64)
    self.bn4 = nn.BatchNorm2d(128)
    self.dropout = nn.Dropout(0.5)

  def forward(self, X):
    x = F.relu(self.bn1(self.conv1(X)))
    x = self.pool1(x)
    x = F.relu(self.bn2(self.conv2(x)))
    x = self.pool2(x)
    x = F.relu(self.bn3(self.conv3(x)))
    x = F.relu(self.bn4(self.conv4(x)))
    x = x.reshape(-1, 8 * 8 * 128)
    x = F.relu(self.fc1(x))
    x = self.dropout(x)
    x = self.fc2(x)
    return x

model = Net(3, 5)
print(model)

In [17]:
#Loss
criterion = nn.CrossEntropyLoss()

#Optimizer
lr = 3e-4
optimizer = torch.optim.Adam(model.parameters(), lr)

#Scheduler: lowers learning rate when the learning process gets stuck after several epochs
n_epochs = 75
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max= n_epochs,   #Total number of epochs
    eta_min=1e-6      #Minimun learning rate
)

In [ ]:
train_losses = []
val_losses = []
lowest_val_loss = 1000
bestModel = Net(3, 5) #Model with the lowest validation loss

for epoch in range(n_epochs):
    train_loss = 0.0
    model.train()
    for img, label in train_loader:
        optimizer.zero_grad()
        output = model(img)
        loss = criterion(output, label)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * img.size(0)

    train_loss = train_loss/len(train_loader.dataset)
    train_losses.append(train_loss)
    #Validation:
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
      for img, label in val_loader:
        output = model(img)
        loss = criterion(output, label)
        val_loss += loss.item() * img.size(0)
    val_loss = val_loss / len(val_loader.dataset)
    scheduler.step()
    val_losses.append(val_loss)
    print('Epoch: {} \tTraining Loss: {:.6f}'.format(epoch+1,train_loss))
    print('\t\tValidation Loss: {:.6f}'.format(val_loss))
    #A reference to the best model found so far is maintained
    if val_loss < lowest_val_loss:
      lowest_val_loss = val_loss
      bestModel = copy.deepcopy(model)

model = bestModel

plt.plot(range(len(train_losses)), train_losses, color='red', label="Train loss")
plt.plot(range(len(val_losses)), val_losses, color='blue', label="Validation loss")
plt.legend()
plt.xlabel('epoch')
plt.ylabel('loss')
plt.grid(True)
plt.text(0.78, 0.8, f"n_epochs = {len(train_losses)}", transform=plt.gca().transAxes)
plt.text(0.78, 0.75, f"init lr = {lr}", transform=plt.gca().transAxes)

In [ ]:
#Testing:
test_loss = 0.0
class_correct = list(0. for i in range(len(classes)))
class_total = list(0. for i in range(len(classes)))

model.eval()

for img, target in test_loader:
    batch_size = img.size(0)
    output = model(img)
    loss = criterion(output, target)
    test_loss += loss.item() * batch_size
    _, pred = torch.max(output, 1)

    correct = pred.eq(target.view_as(pred))
    for i in range(batch_size):
        label = target[i].item()
        class_correct[label] += correct[i].item()
        class_total[label] += 1
test_loss = test_loss/len(test_loader.dataset)

for i in range(len(classes)):
    if class_total[i] > 0:
        print('Test Accuracy of %5s: %2d%% (%2d/%2d)' % (
            classes[i], 100 * class_correct[i] / class_total[i],
            np.sum(class_correct[i]), np.sum(class_total[i])))
    else:
        print('Test Accuracy of %5s: N/A (no training examples)' % (classes[i]))

tloss = 0.0
test_correct = 0
with torch.no_grad():
  for imgs, labels in test_loader:
    outputs = model(imgs)
    loss = criterion(outputs, labels)
    tloss += loss.item() * imgs.size(0)
    _, preds = torch.max(outputs, 1)
    test_correct += torch.sum(preds == labels).item()

  tloss = tloss / len(test_loader.dataset)
  test_acc = 100. * test_correct / len(test_loader.dataset)
  print(f"Train loss = {train_loss:.4f}; Validation loss = {lowest_val_loss:.4f}; Test loss = {tloss:.4f}; Test acc = {test_acc:.2f}%")